<a href="https://colab.research.google.com/github/Samiya-AW/Retrieval-Augmented-LLM-for-Cognitive-Reframing-of-Negative-Thoughts/blob/main/LLM_based_Cognitive_Reframing_for_Negative_Thoughts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Retrieval-Based Cognitive Reframing System**

In [ ]:
!pip install sentence-transformers transformers accelerate

# Install Sentence Transformers for creating text embeddings
!pip install sentence-transformers

In [ ]:
# Load data set
# Prepare dataset for retrieval
# Combine situation + thought

import pandas as pd

url = "https://raw.githubusercontent.com/behavioral-data/Cognitive-Reframing/refs/heads/main/data/reframing_dataset.csv"
df = pd.read_csv(url)

df["combined_text"] = df["situation"] + " " + df["thought"]
df.head()


In [ ]:
# Load embedding model

from sentence_transformers import SentenceTransformer, util
import torch

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Compute embeddings for all thoughts i.e. vectorize the dataset

corpus_embeddings = embed_model.encode(
    df["combined_text"].tolist(),
    convert_to_tensor=True
)

In [ ]:
# Prepare dataset for retrieval
# Combine situation + thought

df["combined_text"] = df["situation"] + " " + df["thought"]

# Compute embeddings for all thoughts i.e. vectorize the dataset
corpus_embeddings = embed_model.encode(df["combined_text"].tolist(), convert_to_tensor=True)

In [ ]:
# Generation of retrieval-augmented prompt with similarity score display

def get_best_match(user_situation, user_thought):
    query = user_situation + " " + user_thought
    query_embedding = embed_model.encode(query, convert_to_tensor=True)

    scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    best_idx = torch.argmax(scores)
    best_score = torch.max(scores)

    return int(best_idx), float(best_score)

In [ ]:
# Generate LLM-based reframe

def retrieve_reframe(user_situation, user_thought):
    idx, score = get_best_match(user_situation, user_thought)
    matched_row = df.iloc[idx]

    return {
        "matched_thought": matched_row["thought"],
        "retrieved_reframe": matched_row["reframe"],
        "similarity_score": score
    }

In [ ]:
# Test the reframe

situation = "I failed my exam."
thought = "I am a complete failure."

result = retrieve_reframe(situation, thought)

print("Most Similar Thought:", result["matched_thought"])
print("Retrieved Rational Reframe:", result["retrieved_reframe"])
print("Similarity Score:", result["similarity_score"])

In [ ]:
# Randomly sample 20 examples for evaluation
test_df = df.sample(20, random_state=42).reset_index(drop=True)

# Evaluate retrieval similarity using leave-one-out matching to avoid self-retrieval bias
def evaluate_retrieval_proper(test_dataframe):
    similarity_scores = []

    for _, row in test_dataframe.iterrows():
        # Build temporary corpus excluding this example
        temp_df = df[
            ~((df["situation"] == row["situation"]) &
              (df["thought"] == row["thought"]))
        ].reset_index(drop=True)

        temp_embeddings = embed_model.encode(
            temp_df["combined_text"].tolist(),
            convert_to_tensor=True
        )

        query = row["situation"] + " " + row["thought"]
        query_embedding = embed_model.encode(query, convert_to_tensor=True)

        scores = util.cos_sim(query_embedding, temp_embeddings)[0]
        best_score = torch.max(scores)

        similarity_scores.append(float(best_score))

    return similarity_scores

# Run evaluation

scores = evaluate_retrieval_proper(test_df)

print("Average Similarity:", sum(scores)/len(scores))
print("Max Similarity:", max(scores))
print("Min Similarity:", min(scores))

In [ ]:
exact_matches = 0

for i, row in test_df.iterrows():
    idx, _ = get_best_match(row["situation"], row["thought"])
    if df.iloc[idx]["thought"] == row["thought"]:
        exact_matches += 1

print("Exact Thought Retrieval Accuracy:", exact_matches / len(test_df))

In [ ]:
# Compute how often similarity > 0.60 to show how often does the system find strongly related thoughts

high_similarity = [s for s in scores if s > 0.60]
print("Proportion above 0.60:", len(high_similarity)/len(scores))

# **LLM-Based Cognitive Reframing System**

In [ ]:
# Install required libraries

!pip install sentence-transformers transformers accelerate

In [ ]:
# Load TinyLlama

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto'
)

In [ ]:
# Modify retrieval to top-2

def get_top_k_matches(user_situation, user_thought, k = 2):
  query = user_situation + " " + user_thought
  query_embedding = embed_model.encode(query, convert_to_tensor=True)

  scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
  top_results = torch.topk(scores, k=k)

  return top_results.indices

In [ ]:
# Build LLM prompt

def build_prompt(user_situation, user_thought):
    indices = get_top_k_matches(user_situation, user_thought, k=2)

    examples = ""
    for idx in indices:
        row = df.iloc[int(idx)]
        examples += f"Distorted Thought: {row['thought']}\n"
        examples += f"Rational Response: {row['reframe']}\n\n"

    prompt = f"""
You are a cognitive behavioral therapist.
Rewrite the distorted thought into a balanced and rational perspective.

Here are examples:

{examples}

Now rewrite:

Distorted Thought: {user_thought}
Rational Response:
"""

    return prompt

In [ ]:
# Generate reframe with TinyLlama

def generate_reframe(user_situation, user_thought):
    prompt = build_prompt(user_situation, user_thought)

    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated response after "Rational Response:"
    if "Rational Response:" in result:
        result = result.split("Rational Response:")[-1].strip()

    return result

In [ ]:
# Test the LLM-based reframe

situation = "I failed my exam."
thought = "I am a complete failure."

print(generate_reframe(situation, thought))

In [ ]:
# Test the LLM-based reframe

situation = "I have a student in class who can't understand concepts taught during lecture."
thought = "I am a complete failure."

print(generate_reframe(situation, thought))

In [ ]:
# Test the LLM-based reframe

situation = "I have a student in class who can't understand concepts taught during lecture."
thought = "I feel like a complete failure as a teacher."

print(generate_reframe(situation, thought))